# Anomaly Detection Model Training

This notebook trains an unsupervised Isolation Forest model for Real-Time UPI Fraud Detection.

The model learns the normal behavioral fingerprint of UPI transactions from engineered features. It does not use fraud labels during training. Labels are loaded only for post-training sanity checks.

## 1. Imports and Project Paths

In [1]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler


PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"

FEATURE_MATRIX_PATH = PROCESSED_DATA_DIR / "feature_matrix.csv"
LABELS_PATH = PROCESSED_DATA_DIR / "labels.csv"

SCALER_PATH = MODELS_DIR / "feature_scaler.pkl"
MODEL_PATH = MODELS_DIR / "isolation_forest_model.pkl"
TRAINING_METADATA_PATH = MODELS_DIR / "training_metadata.pkl"
ANOMALY_SCORES_PATH = PROCESSED_DATA_DIR / "anomaly_scores.csv"

MODELS_DIR.mkdir(parents=True, exist_ok=True)

## 2. Training Configuration

In [2]:
@dataclass(frozen=True)
class ModelTrainingConfig:
    """Isolation Forest training configuration."""

    contamination: float = 0.03
    n_estimators: int = 200
    max_samples: str | int = "auto"
    random_state: int = 42
    n_jobs: int = 1


config = ModelTrainingConfig()
config

ModelTrainingConfig(contamination=0.03, n_estimators=200, max_samples='auto', random_state=42, n_jobs=1)

## 3. Load Engineered Features

In [3]:
def load_training_inputs(feature_path: Path, labels_path: Path) -> tuple[pd.DataFrame, pd.Series]:
    """Load model features and labels produced by the feature engineering notebook."""
    if not feature_path.exists():
        raise FileNotFoundError(f"Feature matrix not found: {feature_path}")
    if not labels_path.exists():
        raise FileNotFoundError(f"Labels file not found: {labels_path}")

    features = pd.read_csv(feature_path)
    labels = pd.read_csv(labels_path).squeeze("columns").astype(int)
    return features, labels


def validate_training_inputs(features: pd.DataFrame, labels: pd.Series) -> None:
    """Validate feature matrix quality before model training."""
    if features.empty:
        raise ValueError("Feature matrix is empty.")
    if len(features) != len(labels):
        raise ValueError("Feature matrix and labels must have the same number of rows.")
    if features.isna().any().any():
        raise ValueError("Feature matrix contains missing values.")
    if not np.isfinite(features.to_numpy()).all():
        raise ValueError("Feature matrix contains infinite values.")


X, y = load_training_inputs(FEATURE_MATRIX_PATH, LABELS_PATH)
validate_training_inputs(X, y)

print(f"Feature matrix shape: {X.shape}")
print(f"Fraud labels available for evaluation only: {y.value_counts().sort_index().to_dict()}")
X.head()

Feature matrix shape: (10000, 6)
Fraud labels available for evaluation only: {0: 9700, 1: 300}


,hour_of_day,day_of_week,transaction_value,is_first_time_receiver,velocity_last_10min,amount_vs_user_avg
0,0,0,1487.51,1,1.0,1.000000
1,0,0,2711.60,1,1.0,1.219036
2,1,0,497.85,1,1.0,0.337527
3,1,0,3050.08,1,1.0,1.551246
4,1,0,5436.65,1,1.0,1.000000


## 4. Scale Features

In [4]:
def scale_features(features: pd.DataFrame) -> tuple[np.ndarray, StandardScaler]:
    """Standardize features so distance-based tree splits are not dominated by amount scale."""
    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(features)
    return scaled_features, scaler


X_scaled, scaler = scale_features(X)

scaled_preview = pd.DataFrame(X_scaled, columns=X.columns).describe().round(2)
scaled_preview

,hour_of_day,day_of_week,transaction_value,is_first_time_receiver,velocity_last_10min,amount_vs_user_avg
count,10000.00,10000.00,10000.00,10000.00,10000.00,10000.00
mean,-0.00,-0.00,0.00,-0.00,0.00,-0.00
std,1.00,1.00,1.00,1.00,1.00,1.00
min,-1.60,-1.50,-0.25,-99.99,-0.02,-1.56
25%,-0.89,-1.00,-0.21,0.01,-0.02,-0.62
50%,-0.03,0.01,-0.17,0.01,-0.02,0.00
75%,0.83,1.01,-0.13,0.01,-0.02,0.44
max,1.70,1.51,8.42,0.01,57.73,11.84


## 5. Train Isolation Forest

In [5]:
def train_isolation_forest(
    scaled_features: np.ndarray,
    config: ModelTrainingConfig,
) -> IsolationForest:
    """Train an unsupervised Isolation Forest model."""
    model = IsolationForest(
        n_estimators=config.n_estimators,
        contamination=config.contamination,
        max_samples=config.max_samples,
        random_state=config.random_state,
        n_jobs=config.n_jobs,
    )
    model.fit(scaled_features)
    return model


isolation_forest = train_isolation_forest(X_scaled, config)

print("Model training complete.")
print(f"Number of trees trained: {isolation_forest.n_estimators}")
print(f"Expected anomaly contamination: {isolation_forest.contamination:.2%}")

Model training complete.
Number of trees trained: 200
Expected anomaly contamination: 3.00%


## 6. Generate Training Anomaly Scores

In [6]:
def score_training_data(model: IsolationForest, scaled_features: np.ndarray) -> pd.DataFrame:
    """Create anomaly scores and default model predictions for inspection."""
    decision_scores = model.decision_function(scaled_features)
    anomaly_scores = -decision_scores
    default_predictions = model.predict(scaled_features)

    return pd.DataFrame(
        {
            "decision_score": decision_scores,
            "anomaly_score": anomaly_scores,
            "default_prediction": default_predictions,
            "is_suspicious_default": (default_predictions == -1).astype(int),
        }
    )


score_dataset = score_training_data(isolation_forest, X_scaled)
score_dataset["is_fraud"] = y.to_numpy()

score_dataset.describe().round(4)

,decision_score,anomaly_score,default_prediction,is_suspicious_default,is_fraud
count,10000.0000,10000.0000,10000.0000,10000.0000,10000.0000
mean,0.1382,-0.1382,0.9400,0.0300,0.0300
std,0.0530,0.0530,0.3412,0.1706,0.1706
min,-0.1402,-0.2034,-1.0000,0.0000,0.0000
25%,0.1190,-0.1735,1.0000,0.0000,0.0000
50%,0.1506,-0.1506,1.0000,0.0000,0.0000
75%,0.1735,-0.1190,1.0000,0.0000,0.0000
max,0.2034,0.1402,1.0000,1.0000,1.0000


## 7. Evaluation Sanity Check

In [7]:
def evaluate_default_threshold(score_dataset: pd.DataFrame) -> None:
    """Evaluate the model's default contamination threshold using labels for reporting only."""
    print("Default Isolation Forest threshold report:")
    print(
        classification_report(
            score_dataset["is_fraud"],
            score_dataset["is_suspicious_default"],
            target_names=["Normal", "Fraud"],
            zero_division=0,
        )
    )
    print("Confusion matrix:")
    print(confusion_matrix(score_dataset["is_fraud"], score_dataset["is_suspicious_default"]))


evaluate_default_threshold(score_dataset)

Default Isolation Forest threshold report:
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00      9700
       Fraud       0.94      0.94      0.94       300

    accuracy                           1.00     10000
   macro avg       0.97      0.97      0.97     10000
weighted avg       1.00      1.00      1.00     10000

Confusion matrix:
[[9682   18]
 [  18  282]]


## 8. Save Model Artifacts

In [8]:
def save_training_artifacts(
    model: IsolationForest,
    scaler: StandardScaler,
    features: pd.DataFrame,
    scores: pd.DataFrame,
    config: ModelTrainingConfig,
) -> dict[str, Path]:
    """Persist model, scaler, metadata, and training scores."""
    metadata: dict[str, Any] = {
        "feature_columns": list(features.columns),
        "training_rows": len(features),
        "model_type": "IsolationForest",
        "config": asdict(config),
    }

    joblib.dump(model, MODEL_PATH)
    joblib.dump(scaler, SCALER_PATH)
    joblib.dump(metadata, TRAINING_METADATA_PATH)
    scores.to_csv(ANOMALY_SCORES_PATH, index=False)

    return {
        "model": MODEL_PATH,
        "scaler": SCALER_PATH,
        "metadata": TRAINING_METADATA_PATH,
        "scores": ANOMALY_SCORES_PATH,
    }


artifact_paths = save_training_artifacts(isolation_forest, scaler, X, score_dataset, config)

for artifact_name, artifact_path in artifact_paths.items():
    print(f"Saved {artifact_name}: {artifact_path}")

Saved model: C:\Users\Prompt\Documents\Real-Time-UPI-Fraud-Detection-System_Notebook\Real-Time-UPI-Fraud-Detection-System\models\isolation_forest_model.pkl
Saved scaler: C:\Users\Prompt\Documents\Real-Time-UPI-Fraud-Detection-System_Notebook\Real-Time-UPI-Fraud-Detection-System\models\feature_scaler.pkl
Saved metadata: C:\Users\Prompt\Documents\Real-Time-UPI-Fraud-Detection-System_Notebook\Real-Time-UPI-Fraud-Detection-System\models\training_metadata.pkl
Saved scores: C:\Users\Prompt\Documents\Real-Time-UPI-Fraud-Detection-System_Notebook\Real-Time-UPI-Fraud-Detection-System\data\processed\anomaly_scores.csv


## 9. Output Summary

Generated files:

- `models/isolation_forest_model.pkl`
- `models/feature_scaler.pkl`
- `models/training_metadata.pkl`
- `data/processed/anomaly_scores.csv`

The next notebook should tune the anomaly threshold using F1-score to balance fraud recall and false positives.